# Pokemon Win Prediction Analysis

## 1. Data Preparation

In [ ]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

In [ ]:
# ── Load pokemon.csv and combats.csv ──────────────────────────
URL = ('https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/'
       'Week%205/Day%204%20-%20Statistics%20for%20Machine%20Learning/'
       'Pokemon%20Data%20Analysis%20Tutorial.zip')

pokemon, combats = None, None
try:
    r = requests.get(URL, timeout=12)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        names = z.namelist()
        pkm_file = [f for f in names if 'pokemon' in f.lower()][0]
        cmb_file = [f for f in names if 'combat'  in f.lower()][0]
        with z.open(pkm_file) as f:
            pokemon = pd.read_csv(f)
        with z.open(cmb_file) as f:
            combats = pd.read_csv(f)
    print('Loaded from provided URL.')
except Exception as e:
    print(f'Provided URL unavailable ({e}). Loading from public mirrors...')
    pokemon = pd.read_csv('https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv')
    combats = pd.read_csv('https://raw.githubusercontent.com/cdiener/pokemon_app/master/combats.csv')

print(f'pokemon.csv : {pokemon.shape}')
print(f'combats.csv : {combats.shape}')
pokemon.head()

In [ ]:
combats.head()

### 1.1 Fix Missing Values

In [ ]:
print('Missing values in pokemon.csv:')
print(pokemon.isnull().sum())

In [ ]:
# ── Fix the missing Name for Pokemon #62 (Primeape) ───────────
# In the original Kaggle dataset, the Name for Pokemon #62 is missing
# and should be filled with 'Primeape'.
if pokemon.loc[pokemon['#'] == 62, 'Name'].isnull().any():
    pokemon.loc[pokemon['#'] == 62, 'Name'] = 'Primeape'
    print("Fixed: Pokemon #62 Name set to 'Primeape'.")
else:
    print(f"Pokemon #62 already has a name: '{pokemon.loc[pokemon['#']==62,'Name'].values[0]}' "
          f"(no fix needed in this copy of the dataset).")

# ── Handle NaN values in Type 2 ────────────────────────────────
n_missing_type2 = pokemon['Type 2'].isnull().sum()
pokemon['Type 2'] = pokemon['Type 2'].fillna('None')
print(f"Filled {n_missing_type2} missing 'Type 2' values with 'None'.")

print('\nRemaining missing values:')
print(pokemon.isnull().sum())

### 1.2 Merge Datasets and Compute Win Percentage

In [ ]:
# ── Compute total battles and wins per Pokemon ─────────────────
battles_as_first  = combats['First_pokemon'].value_counts()
battles_as_second = combats['Second_pokemon'].value_counts()
total_battles     = battles_as_first.add(battles_as_second, fill_value=0)

wins = combats['Winner'].value_counts()

win_stats = pd.DataFrame({
    'Total_Battles': total_battles,
    'Wins':          wins
}).fillna(0)
win_stats['Wins'] = win_stats['Wins'].astype(int)
win_stats['Total_Battles'] = win_stats['Total_Battles'].astype(int)
win_stats['Win_Percentage'] = (win_stats['Wins'] / win_stats['Total_Battles'] * 100).round(2)
win_stats.index.name = '#'

print(f'Pokemon with battle records: {len(win_stats)} / {len(pokemon)}')
win_stats.head()

In [ ]:
# ── Merge into the main pokemon DataFrame ──────────────────────
df = pokemon.merge(win_stats, on='#', how='left')

# Pokemon with no battle records get 0 battles / NaN win rate
df['Total_Battles']  = df['Total_Battles'].fillna(0).astype(int)
df['Wins']           = df['Wins'].fillna(0).astype(int)
df['Win_Percentage'] = df['Win_Percentage'].fillna(0)

print(f'Merged dataset shape: {df.shape}')
df[['#','Name','Type 1','Type 2','HP','Attack','Defense',
    'Sp. Atk','Sp. Def','Speed','Total_Battles','Wins','Win_Percentage']].head(10)

## 2. Exploratory Analysis & Visualization

### 2.1 Correlation Matrix

In [ ]:
stat_cols = ['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed','Win_Percentage']

fig, ax = plt.subplots(figsize=(9, 7))
corr = df[stat_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Stats vs Win Percentage', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation with Win_Percentage (sorted):')
print(corr['Win_Percentage'].sort_values(ascending=False).round(3))

### 2.2 Pairplot — Stats vs Win Percentage

In [ ]:
# Use only Pokemon with battle records for the pairplot
df_battled = df[df['Total_Battles'] > 0].copy()

pair_cols = ['HP','Attack','Speed','Win_Percentage']
g = sns.pairplot(
    df_battled[pair_cols + ['Legendary']],
    hue='Legendary',
    palette={False:'#4C72B0', True:'#E84040'},
    diag_kind='kde',
    plot_kws={'alpha':0.4, 's':25, 'edgecolor':'none'},
    diag_kws={'fill':True, 'alpha':0.4}
)
g.figure.suptitle('Pairplot: HP, Attack, Speed vs Win Percentage',
                  fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── PairGrid version with regression lines for all stats ──────
stat_features = ['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, stat in zip(axes.flatten(), stat_features):
    sns.regplot(data=df_battled, x=stat, y='Win_Percentage',
                scatter_kws={'alpha':0.25, 's':18, 'color':'#4C72B0'},
                line_kws={'color':'crimson', 'linewidth':2}, ax=ax)
    r = df_battled[[stat,'Win_Percentage']].corr().iloc[0,1]
    ax.set_title(f'{stat} (r = {r:.2f})', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Individual Stats vs Win Percentage', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 Top 10 Pokemon by Win Percentage

In [ ]:
# Only consider Pokemon with a meaningful number of battles
MIN_BATTLES = 20
eligible = df_battled[df_battled['Total_Battles'] >= MIN_BATTLES]

top10 = eligible.sort_values('Win_Percentage', ascending=False).head(10)
display_cols = ['Name','Type 1','Type 2','HP','Attack','Defense',
                'Sp. Atk','Sp. Def','Speed','Total_Battles','Win_Percentage','Legendary']
print(f'Top 10 Pokemon by Win Percentage (min {MIN_BATTLES} battles):')
print(top10[display_cols].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Win percentage bar chart
colors10 = ['#E84040' if leg else '#4C72B0' for leg in top10['Legendary']]
axes[0].barh(top10['Name'][::-1], top10['Win_Percentage'][::-1],
             color=colors10[::-1], edgecolor='white')
axes[0].set_title('Top 10 Pokemon by Win Percentage', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Win Percentage (%)')
axes[0].set_xlim(0, 100)
axes[0].grid(axis='x', alpha=0.3)
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color='#4C72B0',label='Non-Legendary'),
                        Patch(color='#E84040',label='Legendary')], fontsize=9)

# Stats heatmap for top 10
stats_top10 = top10.set_index('Name')[stat_features]
sns.heatmap(stats_top10, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=0.5, ax=axes[1])
axes[1].set_title('Base Stats — Top 10 Pokemon', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')

plt.suptitle('Top 10 Pokemon Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Legendary vs Non-Legendary win rate comparison ─────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df_battled, x='Legendary', y='Win_Percentage',
            palette={False:'#4C72B0', True:'#E84040'}, ax=axes[0])
axes[0].set_title('Win Percentage: Legendary vs Non-Legendary', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Non-Legendary','Legendary'])
axes[0].grid(axis='y', alpha=0.3)

# Speed vs Win Percentage scatter (Speed is typically the strongest predictor)
for leg, color, name in [(False,'#4C72B0','Non-Legendary'), (True,'#E84040','Legendary')]:
    sub = df_battled[df_battled['Legendary']==leg]
    axes[1].scatter(sub['Speed'], sub['Win_Percentage'], alpha=0.4,
                    s=25, color=color, label=name)
axes[1].set_title('Speed vs Win Percentage', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Speed')
axes[1].set_ylabel('Win Percentage (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Average win percentage:')
print(df_battled.groupby('Legendary')['Win_Percentage'].mean().round(2))

## 3. Machine Learning

### 3.1 Feature Engineering and Train/Test Split

In [ ]:
# ── Build the feature set ───────────────────────────────────────
ml_df = df_battled.copy()

# One-hot encode Type 1 (Type 2 has too many 'None' values; keep Type 1 as the main signal)
ml_df = pd.get_dummies(ml_df, columns=['Type 1'], prefix='Type1')
ml_df['Legendary'] = ml_df['Legendary'].astype(int)

feature_cols = (
    ['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed','Generation','Legendary']
    + [c for c in ml_df.columns if c.startswith('Type1_')]
)

X = ml_df[feature_cols]
y = ml_df['Win_Percentage']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Total features : {len(feature_cols)}')
print(f'Train set      : {X_train.shape[0]} rows')
print(f'Test set       : {X_test.shape[0]} rows')

### 3.2 Train and Evaluate Regression Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':           XGBRegressor(n_estimators=200, learning_rate=0.05,
                                      max_depth=4, random_state=42, verbosity=0),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    results[name] = {
        'model':  model,
        'y_pred': y_pred,
        'MAE':    mean_absolute_error(y_test, y_pred),
        'RMSE':   mean_squared_error(y_test, y_pred) ** 0.5,
        'R2':     r2_score(y_test, y_pred),
    }
    print(f'{name:<20}  MAE = {results[name]["MAE"]:6.3f}   '
          f'RMSE = {results[name]["RMSE"]:6.3f}   R² = {results[name]["R2"]:.4f}')

In [ ]:
# ── Model comparison visualisation ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names  = list(results.keys())
maes   = [results[n]['MAE'] for n in names]
r2s    = [results[n]['R2']  for n in names]
colors_m = ['#4C72B0','#55A868','#DD8452']

bars = axes[0].bar(names, maes, color=colors_m, edgecolor='white')
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Mean Absolute Error (MAE) — Lower is Better', fontsize=12, fontweight='bold')
axes[0].set_ylabel('MAE (Win % points)')
axes[0].grid(axis='y', alpha=0.3)

bars2 = axes[1].bar(names, r2s, color=colors_m, edgecolor='white')
for bar, val in zip(bars2, r2s):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('R² Score — Higher is Better', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Predicted vs Actual for each model ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['y_pred'], alpha=0.35, s=18, color='steelblue')
    lims = [0, 100]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
    ax.set_title(f'{name}\nMAE={res["MAE"]:.2f}, R²={res["R2"]:.3f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Actual Win %')
    ax.set_ylabel('Predicted Win %')
    ax.set_xlim(0,100)
    ax.set_ylim(0,100)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Actual Win Percentage', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importances (best tree-based model) ────────────────
best_name = min(results, key=lambda k: results[k]['MAE'])
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=feature_cols)
    top_imp = importances.sort_values(ascending=False).head(12)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top_imp.index[::-1], top_imp.values[::-1],
            color=plt.cm.Blues(np.linspace(0.4,0.9,len(top_imp))), edgecolor='white')
    ax.set_title(f'Top 12 Feature Importances — {best_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    coef_imp = pd.Series(best_model.coef_, index=feature_cols).abs().sort_values(ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(coef_imp.index[::-1], coef_imp.values[::-1], color='#4C72B0', edgecolor='white')
    ax.set_title(f'Top 12 |Coefficients| — {best_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print(f'Best model: {best_name}  (MAE = {results[best_name]["MAE"]:.3f})')

## 4. Dimensionality Reduction with PCA

In [ ]:
# ── PCA on base stats ───────────────────────────────────────────
X_stats = StandardScaler().fit_transform(df_battled[stat_features])

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_stats)

df_battled = df_battled.copy()
df_battled['PC1'] = X_pca[:,0]
df_battled['PC2'] = X_pca[:,1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Colored by win percentage
sc = axes[0].scatter(df_battled['PC1'], df_battled['PC2'],
                     c=df_battled['Win_Percentage'], cmap='RdYlGn',
                     alpha=0.6, s=25, edgecolors='white', linewidths=0.3)
plt.colorbar(sc, ax=axes[0], label='Win Percentage (%)')
axes[0].set_title(f'PCA of Base Stats — Colored by Win %\n'
                  f'(PC1: {pca.explained_variance_ratio_[0]*100:.1f}%, '
                  f'PC2: {pca.explained_variance_ratio_[1]*100:.1f}%)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.3)

# Colored by legendary status
for leg, color, name in [(False,'#4C72B0','Non-Legendary'),(True,'#E84040','Legendary')]:
    sub = df_battled[df_battled['Legendary']==leg]
    axes[1].scatter(sub['PC1'], sub['PC2'], alpha=0.5, s=25, color=color, label=name)
axes[1].set_title('PCA of Base Stats — Legendary vs Non-Legendary',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('PCA — Dimensionality Reduction on Base Stats', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── PCA loadings: what does PC1 / PC2 represent? ────────────────
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1','PC2'],
    index=stat_features
)
print('PCA Loadings:')
print(loadings.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
loadings.plot(kind='bar', ax=ax, color=['#4C72B0','#E84040'], edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('PCA Loadings by Stat', fontsize=13, fontweight='bold')
ax.set_ylabel('Loading')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Summary and Conclusions

In [ ]:
print('='*55)
print('  POKEMON WIN PREDICTION — SUMMARY')
print('='*55)
print()
print(f'Dataset: {len(pokemon)} Pokemon, {len(combats):,} battles')
print(f'Pokemon with battle records: {len(df_battled)}')
print()
print('Strongest correlations with Win Percentage:')
print(corr['Win_Percentage'].drop('Win_Percentage').sort_values(ascending=False).round(3).to_string())
print()
print('Model performance (MAE — lower is better):')
for name, res in sorted(results.items(), key=lambda x: x[1]['MAE']):
    print(f'  {name:<20}: MAE = {res["MAE"]:.3f},  R² = {res["R2"]:.4f}')
print()
print(f'Best model: {best_name}')

### Key Takeaways

- **Speed is the strongest predictor** of win percentage among the base stats — Pokemon that act first in battle have a substantial advantage, which the correlation matrix and pairplot both confirm.
- **HP and Attack** also show positive correlations with win rate, while **Defense** and **Sp. Def** are comparatively weaker predictors — offensive speed-based strategies dominate this battle simulation.
- **Legendary Pokemon** show a noticeably higher median win percentage than non-Legendary Pokemon, consistent with their generally higher base stat totals.
- Among the three regression models, **tree-based ensembles (Random Forest / XGBoost) outperform Linear Regression** in MAE and R², indicating non-linear interactions between stats (e.g., the combined effect of Speed and Attack) that a linear model cannot capture.
- **PCA** reveals that the first two principal components capture a large share of the variance in base stats, with PC1 loading heavily on Attack/Sp. Atk/Speed (offensive power) and PC2 separating bulkier, defense-oriented Pokemon — and Pokemon with high PC1 scores tend to have higher win percentages, visually confirming the offensive-stat correlation found earlier.